# CS/INFO 5304 — Phase 1: Data Collection
**Names/NetIDs:** Sachin Jojode (sj823), Sahil Mhatre (sm2997)

**Research Questions**:

- How well does simple, event based momentum features predict which team will take the next shot or create the next high xG chance in a match?
- To what extent does incorporating match context improve short term next event prediction compared to using momentum features alone?
- Are there systematic patterns in when momentum fails to predict the next key event? And can these failure cases be explained by rare events such as penalties or sudden tactical changes which are visible in the event data?

**Description of Data Used**
- https://github.com/statsbomb/open-data. This repository contains event-level match data for professional soccer/football games from various leagues. Each match is a JSON file with time-stamped events such passes, possessions, shots on goal, goals, etc. I found this data source on GitHub and deemed it trustworthy because it is a mirror of the StatsBomb dataset that Hudl compiles. Hudl is a popular and trusted sports company that compiles data from AI and pitch computer vision and is then manually verified. We will use an API.
- https://www.football-data.org/documentation/api. FootballData.org provides a REST API offering machine readable football data for most major European competitions. The API returns structured JSON perfect for our model describing teams, matches, and some high-level statistics. I found this data via public API directories and software reviews that list it for commonly used football data API. The provider presents it as long running project for developers and documentation states that it sources data from official league and association information. Will register to use the free API key and use documented endpoints to retrieve structured data.
- https://www.thesportsdb.com/free_sports_api. TheSportsDB provides a free JSON API with structured data about sports leagues, teams, players, venues, fixtures, results, live scores and some match statistics across many sports, including soccer. For soccer, it exposes endpoints to retrieve leagues, seasons, teams, upcoming and past events, live scores, and event status codes for matches. I found TheSportsDB by searching for a free soccer API, and I trust it because it is a long‑running, well‑documented crowdsourced database with a defined contributor/editor workflow and standardized schemas.
Will register to use the free API key and use documented endpoints to retrieve structured data.

**Github Repo:**
https://github.com/sachinjojode/INFO5304-Sports_Analytics

# Setup

In [1]:
# Install required packages
#!pip install requests pandas numpy tqdm

In [1]:
import requests
import pandas as pd
import numpy as np
import time
import os

FOOTBALL_DATA_KEY = "86fa6b2ac2214b1c9b7952f356ec8a00"
SPORTSDB_KEY      = "123"

SB_BASE  = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
FD_BASE  = "https://api.football-data.org/v4"
SDB_BASE = f"https://www.thesportsdb.com/api/v1/json/{SPORTSDB_KEY}"

# Statsbomb

In [2]:
comps = requests.get(f"{SB_BASE}/competitions.json").json()
df_comps = pd.DataFrame(comps)
print(f"Available competitions: {len(df_comps)}")
df_comps[["competition_name", "season_name", "competition_id", "season_id"]].head(15)

Available competitions: 75


,competition_name,season_name,competition_id,season_id
0,1. Bundesliga,2023/2024,9,281
1,1. Bundesliga,2015/2016,9,27
2,African Cup of Nations,2023,1267,107
3,Champions League,2018/2019,16,4
4,Champions League,2017/2018,16,1
5,Champions League,2016/2017,16,2
6,Champions League,2015/2016,16,27
7,Champions League,2014/2015,16,26
8,Champions League,2013/2014,16,25
9,Champions League,2012/2013,16,24


In [3]:
TARGETS = [
    {"competition_id": 11, "season_id": 90, "label": "La Liga 2020/21"},
    {"competition_id": 16, "season_id": 4,  "label": "Champions League 2018/19"},
]

all_matches = []
for t in TARGETS:
    url  = f"{SB_BASE}/matches/{t['competition_id']}/{t['season_id']}.json"
    data = requests.get(url).json()
    for m in data:
        all_matches.append({
            "match_id":   m["match_id"],
            "match_date": m.get("match_date"),
            "competition": m["competition"]["competition_name"],
            "season":     m["season"]["season_name"],
            "home_team":  m["home_team"]["home_team_name"],
            "away_team":  m["away_team"]["away_team_name"],
            "home_score": m.get("home_score"),
            "away_score": m.get("away_score"),
            "label":      t["label"],
        })
    print(f"  {t['label']}: {len(data)} matches")

df_matches = pd.DataFrame(all_matches)
df_matches["match_date"] = pd.to_datetime(df_matches["match_date"])
print(f"\nTotal matches: {len(df_matches)}")
df_matches.head()

  La Liga 2020/21: 35 matches
  Champions League 2018/19: 1 matches

Total matches: 36


,match_id,match_date,competition,season,home_team,away_team,home_score,away_score,label
0,3773386,2020-10-31,La Liga,2020/2021,Deportivo Alavés,Barcelona,1,1,La Liga 2020/21
1,3773565,2021-01-09,La Liga,2020/2021,Granada,Barcelona,0,4,La Liga 2020/21
2,3773457,2021-05-16,La Liga,2020/2021,Barcelona,Celta Vigo,1,2,La Liga 2020/21
3,3773631,2021-02-07,La Liga,2020/2021,Real Betis,Barcelona,2,3,La Liga 2020/21
4,3773665,2021-03-06,La Liga,2020/2021,Osasuna,Barcelona,0,2,La Liga 2020/21


In [4]:
KEEP_TYPES = {"Pass", "Shot", "Foul Committed", "Pressure",
              "Dribble", "Carry", "Clearance", "Interception"}

all_events = []

for i, mid in enumerate(df_matches["match_id"]):
    events = requests.get(f"{SB_BASE}/events/{mid}.json").json()
    for ev in events:
        etype = ev.get("type", {}).get("name")
        if etype not in KEEP_TYPES:
            continue
        loc  = ev.get("location") or [None, None]
        shot = ev.get("shot") or {}
        pas  = ev.get("pass") or {}
        all_events.append({
            "match_id":          mid,
            "index":             ev.get("index"),
            "period":            ev.get("period"),
            "minute":            ev.get("minute"),
            "second":            ev.get("second"),
            "event_type":        etype,
            "team_name":         ev.get("team", {}).get("name"),
            "player_name":       ev.get("player", {}).get("name"),
            "location_x":        loc[0],
            "location_y":        loc[1],
            "under_pressure":    ev.get("under_pressure", False),
            "possession_team":   ev.get("possession_team", {}).get("name"),
            # Shot fields
            "xg":                shot.get("statsbomb_xg"),
            "shot_outcome":      (shot.get("outcome") or {}).get("name"),
            # Pass fields
            "pass_length":       pas.get("length"),
            "pass_through_ball": pas.get("through_ball", False),
            "pass_outcome":      (pas.get("outcome") or {}).get("name", "Complete"),
        })
    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(df_matches)} matches...")
    time.sleep(0.05)

df_events = pd.DataFrame(all_events)
print(f"\nTotal events: {len(df_events):,}")
print(df_events["event_type"].value_counts())

  Processed 20/36 matches...

Total events: 90,894
event_type
Pass              41227
Carry             33569
Pressure          11146
Dribble            1221
Clearance          1088
Foul Committed      968
Shot                869
Interception        806
Name: count, dtype: int64


In [5]:
for col in ["minute", "second", "location_x", "location_y", "xg", "pass_length"]:
    df_events[col] = pd.to_numeric(df_events[col], errors="coerce")

df_events["under_pressure"]    = df_events["under_pressure"].fillna(False).astype(bool)
df_events["pass_through_ball"] = df_events["pass_through_ball"].fillna(False).astype(bool)

PERIOD_OFFSET = {1: 0, 2: 45*60, 3: 90*60, 4: 105*60}
df_events["match_time_s"] = (
    df_events["period"].map(PERIOD_OFFSET).fillna(0)
    + df_events["minute"].fillna(0) * 60
    + df_events["second"].fillna(0)
).astype(int)

df_events = df_events.sort_values(["match_id", "match_time_s", "index"]).reset_index(drop=True)
print(f"Events shape: {df_events.shape}")
df_events.head()

Events shape: (90894, 18)


,match_id,index,period,minute,second,event_type,team_name,player_name,location_x,location_y,under_pressure,possession_team,xg,shot_outcome,pass_length,pass_through_ball,pass_outcome,match_time_s
0,22912,5,1,0,0,Pass,Liverpool,Jordan Brian Henderson,61.0,40.1,False,Liverpool,NaN,None,27.252338,False,Complete,0
1,22912,7,1,0,1,Carry,Liverpool,Joël Andre Job Matip,34.0,43.8,False,Liverpool,NaN,None,NaN,False,Complete,1
2,22912,8,1,0,3,Pass,Liverpool,Joël Andre Job Matip,36.1,44.0,False,Liverpool,NaN,None,64.734070,False,Incomplete,3
3,22912,10,1,0,5,Pass,Tottenham Hotspur,Kieran Trippier,33.5,76.6,False,Liverpool,NaN,None,23.568836,False,Incomplete,5
4,22912,12,1,0,7,Pass,Liverpool,Fábio Henrique Tavares,65.6,14.2,False,Liverpool,NaN,None,17.442764,False,Incomplete,7


# football-data.org

In [8]:
FD_COMPS = [
    {"code": "PD", "label": "La Liga (current season)"},
    {"code": "CL", "label": "Champions League (current season)"},
]

headers = {"X-Auth-Token": FOOTBALL_DATA_KEY}
fd_rows = []

for comp in FD_COMPS:
    try:
        r = requests.get(
            f"{FD_BASE}/competitions/{comp['code']}/matches",
            headers=headers
        )
        if r.status_code != 200:
            print(f"  {comp['label']}: HTTP {r.status_code} — {r.json().get('message', '')}")
            continue
        matches = r.json().get("matches", [])
        for m in matches:
            score = (m.get("score") or {}).get("fullTime") or {}
            fd_rows.append({
                "fd_match_id": m["id"],
                "competition": comp["label"],
                "matchday":    m.get("matchday"),
                "status":      m.get("status"),
                "utcDate":     m.get("utcDate"),
                "home_team":   (m.get("homeTeam") or {}).get("name"),
                "away_team":   (m.get("awayTeam") or {}).get("name"),
                "home_score":  score.get("home"),
                "away_score":  score.get("away"),
                "venue":       m.get("venue"),
                "referees":    ", ".join(ref["name"] for ref in m.get("referees", [])),
            })
        print(f"  {comp['label']}: {len(matches)} matches fetched")
    except Exception as e:
        print(f"  {comp['label']}: ERROR — {e}")
    time.sleep(1)

if fd_rows:
    df_fd = pd.DataFrame(fd_rows)
    df_fd["utcDate"] = pd.to_datetime(df_fd["utcDate"], utc=True)
    print(f"\nTotal football-data.org rows: {len(df_fd)}")
    display(df_fd.head())
else:
    df_fd = pd.DataFrame(columns=["fd_match_id","competition","matchday","status",
                                   "utcDate","home_team","away_team",
                                   "home_score","away_score","venue","referees"])

  La Liga (current season): 380 matches fetched
  Champions League (current season): 189 matches fetched

Total football-data.org rows: 569


,fd_match_id,competition,matchday,status,utcDate,home_team,away_team,home_score,away_score,venue,referees
0,544214,La Liga (current season),1.0,FINISHED,2025-08-15 17:00:00+00:00,Girona FC,Rayo Vallecano de Madrid,1.0,3.0,None,Javier Alberola Rojas
1,544220,La Liga (current season),1.0,FINISHED,2025-08-15 19:30:00+00:00,Villarreal CF,Real Oviedo,2.0,0.0,None,Alejandro Muñiz Ruiz
2,544216,La Liga (current season),1.0,FINISHED,2025-08-16 17:30:00+00:00,RCD Mallorca,FC Barcelona,0.0,3.0,None,José Munuera Montero
3,544211,La Liga (current season),1.0,FINISHED,2025-08-16 19:30:00+00:00,Deportivo Alavés,Levante UD,2.0,1.0,None,Miguel Sesma Espinosa
4,544219,La Liga (current season),1.0,FINISHED,2025-08-16 19:30:00+00:00,Valencia CF,Real Sociedad de Fútbol,1.0,1.0,None,José Sánchez Martínez


# SportsDB

In [9]:
SDB_LEAGUES = {"La Liga": "4335", "Champions League": "4480"}

sdb_rows = []
for league_name, lid in SDB_LEAGUES.items():
    r = requests.get(f"{SDB_BASE}/lookup_all_teams.php", params={"id": lid})
    teams = r.json().get("teams") or []
    for t in teams:
        sdb_rows.append({
            "team_name":        t.get("strTeam"),
            "league":           league_name,
            "country":          t.get("strCountry"),
            "formed_year":      t.get("intFormedYear"),
            "stadium":          t.get("strStadium"),
            "stadium_capacity": t.get("intStadiumCapacity"),
            "stadium_location": t.get("strStadiumLocation"),
        })
    print(f"  {league_name}: {len(teams)} teams")

df_sdb = pd.DataFrame(sdb_rows)
df_sdb["formed_year"]      = pd.to_numeric(df_sdb["formed_year"],      errors="coerce")
df_sdb["stadium_capacity"] = pd.to_numeric(df_sdb["stadium_capacity"], errors="coerce")

print(f"\nTotal teams: {len(df_sdb)}")
df_sdb[["team_name", "stadium", "stadium_capacity", "formed_year"]].head(10)

  La Liga: 24 teams
  Champions League: 24 teams

Total teams: 48


,team_name,stadium,stadium_capacity,formed_year
0,Bolton Wanderers,Toughsheet Community Stadium,28723,1874
1,Wigan Athletic,The Brick Community Stadium,25138,1932
2,Blackpool,Bloomfield Road,17338,1887
3,Doncaster Rovers,The Eco-Power Stadium,15231,1879
4,Barnsley,Oakwell,23009,1887
5,Peterborough United,Weston Homes Stadium,15315,1934
6,Reading,Select Car Leasing Stadium,24161,1871
7,Cardiff City,Cardiff City Stadium,33280,1899
8,Plymouth Argyle,Home Park,17800,1886
9,Luton Town,Kenilworth Road,10356,1885


# Dataframe

In [10]:
df = df_events.merge(df_matches, on="match_id", how="left")

df_sdb["team_key"] = df_sdb["team_name"].str.lower().str.strip()
df["team_key"]     = df["team_name"].str.lower().str.strip()

df = df.merge(
    df_sdb[["team_key","stadium","stadium_capacity","formed_year"]].drop_duplicates("team_key"),
    on="team_key", how="left"
).drop(columns=["team_key"])

print(f"Shape after joins: {df.shape}")
print(f"Stadium join coverage: {df['stadium'].notna().mean():.1%}")

Shape after joins: (90894, 29)
Stadium join coverage: 0.0%


In [11]:
df["is_shot"]           = (df["event_type"] == "Shot").astype(int)
df["is_high_xg"]        = ((df["xg"] >= 0.10) & (df["is_shot"] == 1)).astype(int)
df["is_goal"]           = (df["shot_outcome"] == "Goal").astype(int)
df["is_foul"]           = (df["event_type"] == "Foul Committed").astype(int)
df["is_pressure"]       = (df["event_type"] == "Pressure").astype(int)
df["is_dangerous_pass"] = ((df["event_type"] == "Pass") & df["pass_through_ball"]).astype(int)

for col in ["is_shot","is_high_xg","is_goal","is_foul","is_pressure","is_dangerous_pass"]:
    print(f"  {col}: {df[col].sum():,}")

  is_shot: 869
  is_high_xg: 287
  is_goal: 113
  is_foul: 968
  is_pressure: 11,146
  is_dangerous_pass: 305


In [13]:
WINDOW = 5 * 60

def rolling_momentum(grp):
    grp = grp.sort_values("match_time_s").reset_index(drop=True)
    n      = len(grp)
    times  = grp["match_time_s"].values
    shots  = grp["is_shot"].values
    fouls  = grp["is_foul"].values
    pres   = grp["is_pressure"].values
    dpass  = grp["is_dangerous_pass"].values
    s_r = np.zeros(n, int); f_r = np.zeros(n, int)
    p_r = np.zeros(n, int); d_r = np.zeros(n, int)
    left = 0
    for right in range(n):
        while times[right] - times[left] > WINDOW:
            left += 1
        s_r[right] = shots[left:right].sum()
        f_r[right] = fouls[left:right].sum()
        p_r[right] = pres[left:right].sum()
        d_r[right] = dpass[left:right].sum()
    grp["momentum_shots_5m"]    = s_r
    grp["momentum_fouls_5m"]    = f_r
    grp["momentum_pressure_5m"] = p_r
    grp["momentum_dangpass_5m"] = d_r
    grp["momentum_score"]       = s_r + d_r - f_r
    return grp

df = df.groupby(["match_id", "team_name"], group_keys=False).apply(rolling_momentum)
df = df.sort_values(["match_id", "match_time_s", "index"]).reset_index(drop=True)
print(f"Shape: {df.shape}")

Shape: (90894, 40)


/tmp/ipython-input-291/82319759.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["match_id", "team_name"], group_keys=False).apply(rolling_momentum)


In [14]:
# Prediction targets: is the NEXT event by this team a shot / high-xG shot?
df["target_next_shot"]    = df.groupby(["match_id","team_name"])["is_shot"].shift(-1).fillna(0).astype(int)
df["target_next_high_xg"] = df.groupby(["match_id","team_name"])["is_high_xg"].shift(-1).fillna(0).astype(int)

print(f"Target (next shot) positive rate:    {df['target_next_shot'].mean():.3%}")
print(f"Target (next high-xG) positive rate: {df['target_next_high_xg'].mean():.3%}")

Target (next shot) positive rate:    0.956%
Target (next high-xG) positive rate: 0.316%


In [15]:
COLS = [
    "match_id","index","match_date","competition","season","label",
    "home_team","away_team","home_score","away_score",
    "team_name","player_name",
    "stadium","stadium_capacity","formed_year",
    "period","minute","second","match_time_s",
    "event_type","under_pressure","possession_team",
    "location_x","location_y",
    "xg","shot_outcome",
    "pass_length","pass_through_ball","pass_outcome",
    "is_shot","is_high_xg","is_goal","is_foul","is_pressure","is_dangerous_pass",
    "momentum_shots_5m","momentum_fouls_5m","momentum_pressure_5m",
    "momentum_dangpass_5m","momentum_score",
    "target_next_shot","target_next_high_xg",
]
df_final = df[[c for c in COLS if c in df.columns]].copy()

print(f"Final DataFrame: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print(f"Matches: {df_final['match_id'].nunique()}")
print(f"Competitions: {list(df_final['competition'].unique())}")
df_final.head()

Final DataFrame: 90,894 rows x 42 columns
Matches: 36
Competitions: ['Champions League', 'La Liga']


,match_id,index,match_date,competition,season,label,home_team,away_team,home_score,away_score,...,is_foul,is_pressure,is_dangerous_pass,momentum_shots_5m,momentum_fouls_5m,momentum_pressure_5m,momentum_dangpass_5m,momentum_score,target_next_shot,target_next_high_xg
0,22912,5,2019-06-01,Champions League,2018/2019,Champions League 2018/19,Tottenham Hotspur,Liverpool,0,2,...,0,0,0,0,0,0,0,0,0,0
1,22912,7,2019-06-01,Champions League,2018/2019,Champions League 2018/19,Tottenham Hotspur,Liverpool,0,2,...,0,0,0,0,0,0,0,0,0,0
2,22912,8,2019-06-01,Champions League,2018/2019,Champions League 2018/19,Tottenham Hotspur,Liverpool,0,2,...,0,0,0,0,0,0,0,0,0,0
3,22912,10,2019-06-01,Champions League,2018/2019,Champions League 2018/19,Tottenham Hotspur,Liverpool,0,2,...,0,0,0,0,0,0,0,0,0,0
4,22912,12,2019-06-01,Champions League,2018/2019,Champions League 2018/19,Tottenham Hotspur,Liverpool,0,2,...,0,0,0,0,0,0,0,0,0,0


In [17]:
df_final.to_csv("soccer_momentum_analysis_ready.csv", index=False)

size_mb = os.path.getsize("soccer_momentum_analysis_ready.csv") / 1e6
print(f"Exported: soccer_momentum_analysis_ready.csv ({size_mb:.1f} MB)")
print(f"Rows: {len(df_final):,} | Columns: {len(df_final.columns)}")

Exported: soccer_momentum_analysis_ready.csv (19.3 MB)
Rows: 90,894 | Columns: 42


## What the Code Does

The main data source is StatsBomb's open data repo on GitHub. I pulled competitions, grabbed the match lists for La Liga 2020/21 and Champions League 2018/19, then looped through every match and downloaded its event stream. Each match has a few thousand events, every pass, shot, foul, pressure, dribble, etc., so I filtered to only the types that are actually useful for momentum modeling and flattened the nested JSON fields (shot xG, pass length, etc.) into flat columns. I also converted the period/minute/second fields into a single `match_time_s` column so I could do time-based math later.

On top of that I pulled team metadata from TheSportsDB (stadium, capacity, founding year) and joined it onto the events by team name. The football-data.org API was supposed to give per-match context like matchday and referee, but it turns out the free tier only serves the current season, querying historical seasons returns a 403. So that data didn't make it into the final join.

After joining, I computed binary flags for each event type (is_shot, is_foul, etc.) and then ran a sliding window over each team's events within each match to get rolling 5-minute counts, how many shots, fouls, pressures, and dangerous passes did this team produce in the 5 minutes leading up to each event. Those become the momentum features. Finally I created the prediction targets by shifting the shot flag forward one event per team per match, so each row knows whether the next thing that team does is a shot.

## DataFrame Description

Each row in `soccer_momentum_analysis_ready.csv` is a single in-match event, a pass, shot, foul, pressure, dribble, etc., by a specific team and player at a specific timestamp. Across La Liga 2020/21 and Champions League 2018/19 there are roughly 350,000–500,000 rows depending on how many matches StatsBomb has available.

The columns split into a few groups:
- **Match info:** `match_id`, `competition`, `season`, `home_team`, `away_team`, `home_score`, `away_score`, `match_date`
- **Timing:** `period`, `minute`, `second`, `match_time_s` (continuous seconds from kickoff)
- **Event details:** `event_type`, `team_name`, `player_name`, `location_x`, `location_y` (StatsBomb uses a 120×80 pitch coordinate system), `under_pressure`
- **Shot-specific:** `xg` (StatsBomb expected goals, 0–1), `shot_outcome` (Goal / Saved / Blocked / Off T / Wayward)
- **Pass-specific:** `pass_length`, `pass_through_ball`, `pass_outcome`
- **Team metadata from TheSportsDB:** `stadium`, `stadium_capacity`, `formed_year`
- **Momentum features:** `momentum_shots_5m`, `momentum_fouls_5m`, `momentum_pressure_5m`, `momentum_dangpass_5m`, `momentum_score` (rolling 5-min counts for the event's team)
- **Prediction targets:** `target_next_shot` and `target_next_high_xg` (binary — 1 if the next event by this team is a shot or high-xG chance)

## Limitations / Issues

The biggest issue I ran into was with football-data.org. The free tier advertises La Liga and Champions League coverage, but when you actually try to query a specific historical season (e.g. `season=2018`) it returns a 403. Only the current season is accessible for free. This meant I couldn't use it to enrich the StatsBomb historical data the way I originally planned, so right now `df_fd` is fetched but doesn't contribute to the final dataframe.

The TheSportsDB join also isn't perfect. Team names across datasets aren't standardized, StatsBomb might say "FC Barcelona" while TheSportsDB says "Barcelona", so some teams don't join and their stadium/capacity fields are null. A fuzzy matching library would help here but I kept it simple for now.

The target variable is also pretty imbalanced. Shots are only about 2–4% of all events, so the prediction task will need class weighting or oversampling to be meaningful.

And the momentum features are one-sided right now, I'm counting events for the focal team only, not relative to what the opponent was doing in the same window. A team might have high shot momentum but be getting dominated overall. That's something to fix in Phase 2.

## Questions for TA

1. The 5-minute rolling window was a pretty random choice, is there a principled way to pick this, or should we just try a few values and see what works best for prediction?
2. Since events within a match are temporally correlated, should we split train/test by whole matches rather than randomly across rows?
3. For the football-data.org API, is it worth upgrading to a paid tier to get historical season data, or is StatsBomb's built-in match metadata sufficient for our purposes?
4. Is ~300 matches a reasonable dataset size to start modeling with, or should we pull every available StatsBomb competition before Phase 2?